# YOLO11 inference — Kaggle + Hugging Face

Notebook chứa sẵn phiên bản infer_yolo.py tại thời điểm tạo notebook, không cần upload file .py riêng.
Nếu sửa script sau này, cần cập nhật cell ghi script hoặc thay bằng script mới.

1. Import notebook này vào Kaggle, bật Internet và GPU.
2. Upload **best.pt** thành Kaggle Dataset private và Add Input vào notebook.
3. Thêm secret **HF_TOKEN**, cấp quyền đọc bốn repo và bật quyền truy cập secret cho notebook.
4. Chạy các cell theo thứ tự. Sửa WEIGHTS, VEHICLE_CLASSES sau khi xem model.names.
5. Mặc định benchmark 1.000 ảnh repo đầu tiên. Khi ổn, đặt LIMIT=None để chạy toàn bộ;
   chọn từng repo hoặc lọc camera/ngày để vừa thời gian và ổ đĩa.

Đầu vào hỗ trợ ảnh rời camera_id/YYYYmmdd_HHMMSS.jpg. Nếu repo chỉ chứa Parquet/ZIP,
cell kiểm tra sẽ dừng; cần thêm bước giải mã/giải nén theo schema thực tế.


In [ ]:
%pip install "ultralytics>=8.3,<8.4" "Pillow>=10,<13" "huggingface_hub>=0.23"


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, fnmatch
import torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, snapshot_download
from ultralytics import YOLO

assert sys.version_info >= (3, 11), "Script cần Python >=3.11"
assert torch.cuda.is_available(), "Bật GPU trong Notebook Settings trước"
print("GPU:", torch.cuda.get_device_name(0))
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=os.environ["HF_TOKEN"])
print("Các file weights đang được gắn vào notebook:")
for p in Path('/kaggle/input').rglob('*.pt'):
    print(p)


In [ ]:
# Sửa đường dẫn theo danh sách cell trước.
WEIGHTS = Path('/kaggle/input/YOUR-WEIGHTS-DATASET/best.pt')
assert WEIGHTS.is_file(), "Sửa WEIGHTS thành đường dẫn best.pt đã upload"
model = YOLO(str(WEIGHTS))
print("Class mapping:", model.names)
del model


In [ ]:
ALL_REPOS = [
    'ericng17/hcm_traffic_snapshots',
    'ericng17/hcm_traffic_snapshots2',
    'ericng17/hcm_traffic_snapshots3',
    'ericng17/hcm_traffic_snapshots4',
]
REPOS = ALL_REPOS[:1]  # Chọn từng repo trước; dùng ALL_REPOS khi đủ tài nguyên.
VEHICLE_CLASSES = None  # Ví dụ [0, 1, 2, 3], CHỈ nếu mapping ở trên đều là phương tiện.
LIMIT = 1000            # None = tất cả ảnh phù hợp trong các repo đã chọn.
BATCH = 4
IMAGE_PATTERN = '*'     # Ví dụ 'snapshots/CAMERA_ID/20261112_*.jpg'
TIMEZONE = 'Asia/Ho_Chi_Minh'  # Đổi UTC nếu timestamp lúc crawl là UTC.
OUTPUT_ROOT = Path('/kaggle/working/yolo-results')
DOWNLOAD_ROOT = Path('/kaggle/working/traffic-downloads')
assert VEHICLE_CLASSES, "Điền VEHICLE_CLASSES theo model.names"


## Ghi script .py vào notebook runtime
Cell này thay cho việc upload infer_yolo.py riêng.

In [ ]:
SCRIPT = Path('/kaggle/working/infer_yolo.py')
SCRIPT.write_text('"""Resumable YOLO detection: snapshots/<camera_id>/YYYYmmdd_HHMMSS.jpg.\n\nOne process per GPU, one SQLite output per shard. No network side effects.\n"""\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport logging\nimport sqlite3\nimport time\nfrom collections import Counter\nfrom datetime import datetime\nfrom pathlib import Path\nfrom zoneinfo import ZoneInfo\n\nLOG = logging.getLogger(__name__)\nEXTENSIONS = {\'.jpg\', \'.jpeg\', \'.png\'}\n\n\ndef image_key(path, root, timezone):\n    relative = path.relative_to(root).as_posix()\n    timestamp = datetime.strptime(path.stem, \'%Y%m%d_%H%M%S\').replace(\n        tzinfo=ZoneInfo(timezone)).isoformat()\n    return relative, path.parent.name, timestamp\n\n\ndef shard_for(key, shards):\n    return int(hashlib.sha256(key.encode()).hexdigest(), 16) % shards\n\n\ndef open_database(path, config):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    db = sqlite3.connect(path)\n    db.execute(\'CREATE TABLE IF NOT EXISTS metadata (key TEXT PRIMARY KEY, value TEXT)\')\n    db.execute(\'\'\'CREATE TABLE IF NOT EXISTS images (\n        path TEXT PRIMARY KEY, camera_id TEXT, timestamp TEXT,\n        status TEXT, total INTEGER, counts TEXT, error TEXT,\n        size INTEGER, mtime_ns INTEGER)\'\'\')\n    value = json.dumps(config, sort_keys=True)\n    old = db.execute("SELECT value FROM metadata WHERE key=\'config\'").fetchone()\n    if old and old[0] != value:\n        db.close()\n        raise ValueError(\'Output configuration differs. Use a new --output directory.\')\n    db.execute("INSERT OR IGNORE INTO metadata VALUES (\'config\', ?)", (value,))\n    db.commit()\n    return db\n\n\ndef export_csv(db, destination, names):\n    """Stream an atomic CSV export; errors have blank counts, not zero vehicles."""\n    destination = Path(destination)\n    temporary = destination.with_suffix(\'.csv.tmp\')\n    ids = sorted(names)\n    with temporary.open(\'w\', newline=\'\', encoding=\'utf-8\') as f:\n        writer = csv.writer(f)\n        writer.writerow([\'path\', \'camera_id\', \'timestamp\', \'status\', \'total_vehicles\'] +\n                        [f\'count_{i}_{names[i]}\' for i in ids] + [\'error\'])\n        for row in db.execute(\'SELECT path,camera_id,timestamp,status,total,counts,error \'\n                              \'FROM images ORDER BY camera_id,timestamp,path\'):\n            counts = json.loads(row[5]) if row[5] else {}\n            writer.writerow(list(row[:5]) +\n                            [counts.get(str(i), 0) if row[3] == \'ok\' else \'\' for i in ids] +\n                            [row[6]])\n    temporary.replace(destination)\n\n\ndef process_batch(model, records, db, args, class_ids, device):\n    from PIL import Image\n    images, valid = [], []\n    for record in records:\n        path, key, camera, timestamp, size, mtime = record\n        try:\n            with Image.open(path) as image:\n                image.load()\n                images.append(image.convert(\'RGB\'))\n            valid.append(record)\n        except (OSError, ValueError) as exc:\n            db.execute(\'INSERT OR REPLACE INTO images VALUES (?,?,?,?,?,?,?,?,?)\',\n                       (key, camera, timestamp, \'error\', None, None,\n                        f\'{type(exc).__name__}: {exc}\', size, mtime))\n    if valid:\n        try:\n            results = list(model.predict(source=images, device=device, imgsz=args.imgsz,\n                conf=args.conf, iou=args.iou, classes=class_ids, max_det=args.max_det,\n                half=args.half, verbose=False, save=False, stream=True))\n        except Exception:\n            db.rollback()\n            # Never silently mark OOM/runtime failures as corrupt input images.\n            raise\n        if len(results) != len(valid):\n            db.rollback()\n            raise RuntimeError(\'YOLO result count differs from input count\')\n        for record, result in zip(valid, results):\n            _, key, camera, timestamp, size, mtime = record\n            if result.boxes is None:\n                raise RuntimeError(\'Expected detection boxes; check model task\')\n            counts = Counter(int(x) for x in result.boxes.cls.cpu().tolist())\n            db.execute(\'INSERT OR REPLACE INTO images VALUES (?,?,?,?,?,?,?,?,?)\',\n                       (key, camera, timestamp, \'ok\', sum(counts.values()),\n                        json.dumps(counts), None, size, mtime))\n    db.commit()\n    return len(valid)\n\n\ndef parser():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--source\', type=Path, required=True, help=\'Directory containing camera folders\')\n    p.add_argument(\'--weights\', type=Path, default=Path(\'src/models/yolo/best.pt\'))\n    p.add_argument(\'--output\', type=Path, required=True)\n    p.add_argument(\'--device\', default=\'auto\', help=\'auto (GPU required), 0, 1, or explicit cpu\')\n    p.add_argument(\'--batch\', type=int, default=8)\n    p.add_argument(\'--imgsz\', type=int, default=1024)\n    p.add_argument(\'--conf\', type=float, default=0.25)\n    p.add_argument(\'--iou\', type=float, default=0.7)\n    p.add_argument(\'--max-det\', type=int, default=1000)\n    p.add_argument(\'--half\', action=\'store_true\', help=\'FP16 on CUDA only\')\n    p.add_argument(\'--classes\', type=int, nargs=\'+\', required=True,\n                   help=\'Vehicle class IDs from your custom model, NOT assumed COCO IDs\')\n    p.add_argument(\'--timezone\', default=\'Asia/Ho_Chi_Minh\', help=\'Timezone of filename timestamps\')\n    p.add_argument(\'--num-shards\', type=int, default=1)\n    p.add_argument(\'--shard-index\', type=int, default=0)\n    p.add_argument(\'--limit\', type=int, help=\'Max pending images this invocation, useful for benchmarks\')\n    p.add_argument(\'--retry-errors\', action=\'store_true\')\n    p.add_argument(\'--export-only\', action=\'store_true\', help=\'Export committed results without inference\')\n    return p\n\n\ndef main():\n    args = parser().parse_args()\n    if args.batch < 1 or args.imgsz < 1 or args.max_det < 1:\n        raise ValueError(\'batch, imgsz and max-det must be positive\')\n    if not 0 <= args.shard_index < args.num_shards:\n        raise ValueError(\'Require 0 <= shard-index < num-shards\')\n    if not 0 <= args.conf <= 1 or not 0 <= args.iou <= 1:\n        raise ValueError(\'conf and iou must be between 0 and 1\')\n    if args.limit is not None and args.limit < 1:\n        raise ValueError(\'limit must be positive\')\n    if not args.source.is_dir() or not args.weights.is_file():\n        raise ValueError(\'source directory or local weights missing\')\n    ZoneInfo(args.timezone)\n    import torch\n    import ultralytics\n    from ultralytics import YOLO\n    device = args.device\n    if device == \'auto\':\n        if not torch.cuda.is_available() and not args.export_only:\n            raise RuntimeError(\'No CUDA GPU. Enable cloud GPU or explicitly pass --device cpu.\')\n        device = \'0\'\n    if args.half and device == \'cpu\':\n        raise ValueError(\'--half requires CUDA\')\n    model = YOLO(str(args.weights))\n    if model.task != \'detect\':\n        raise ValueError(\'Expected a detection model\')\n    names = {int(k): v for k, v in model.names.items()}\n    class_ids = sorted(set(args.classes))\n    if not set(class_ids) <= names.keys():\n        raise ValueError(f\'Unknown class IDs. Model classes: {names}\')\n    selected = {i: names[i] for i in class_ids}\n    LOG.info(\'Vehicle classes: %s; device=%s\', selected, device)\n    with args.weights.open(\'rb\') as weights_file:\n        weights_hash = hashlib.file_digest(weights_file, \'sha256\').hexdigest()\n    config = dict(weights_sha256=weights_hash,\n                  classes=selected, imgsz=args.imgsz, conf=args.conf, iou=args.iou,\n                  max_det=args.max_det, half=args.half, timezone=args.timezone,\n                  num_shards=args.num_shards, shard_index=args.shard_index,\n                  ultralytics=ultralytics.__version__, torch=torch.__version__, roi=\'full_frame\')\n    output = args.output\n    db = open_database(output / \'counts.sqlite\', config)\n    (output / \'run.json\').write_text(json.dumps(config, indent=2), encoding=\'utf-8\')\n    started, completed, scheduled, discovered = time.monotonic(), 0, 0, 0\n    try:\n        if not args.export_only:\n            batch = []\n            for path in args.source.rglob(\'*\'):\n                if not path.is_file() or path.suffix.lower() not in EXTENSIONS:\n                    continue\n                discovered += 1\n                key, camera, timestamp = image_key(path, args.source, args.timezone)\n                if shard_for(key, args.num_shards) != args.shard_index:\n                    continue\n                stat = path.stat()\n                old = db.execute(\'SELECT status,size,mtime_ns FROM images WHERE path=?\', (key,)).fetchone()\n                if old and (old[1], old[2]) != (stat.st_size, stat.st_mtime_ns):\n                    raise ValueError(f\'Input changed since previous run: {key}; use a new output\')\n                if old and (old[0] == \'ok\' or not args.retry_errors):\n                    continue\n                batch.append((path, key, camera, timestamp, stat.st_size, stat.st_mtime_ns))\n                scheduled += 1\n                if len(batch) == args.batch:\n                    completed += process_batch(model, batch, db, args, class_ids, device)\n                    batch = []\n                    LOG.info(\'Processed %d valid images (%.2f images/s)\', completed,\n                             completed / max(time.monotonic() - started, .001))\n                if args.limit and scheduled >= args.limit:\n                    break\n            if batch:\n                completed += process_batch(model, batch, db, args, class_ids, device)\n            if not discovered:\n                raise ValueError(\'No images found under source\')\n    finally:\n        db.rollback()\n        export_csv(db, output / \'counts.csv\', selected)\n        LOG.info(\'Committed results: %s\', db.execute(\n            \'SELECT status,count(*) FROM images GROUP BY status\').fetchall())\n        db.close()\n\n\nif __name__ == \'__main__\':\n    logging.basicConfig(level=logging.INFO, format=\'%(asctime)s %(levelname)s %(message)s\')\n    main()\n', encoding='utf-8')
print('Đã tạo', SCRIPT)


## Tải và inference

Pin commit Hugging Face vào hf-source.json để các lần chạy cùng output dùng cùng phiên bản dữ liệu.
Mặc định chỉ tải tối đa LIMIT ảnh, nên benchmark không tải cả repo.
Chạy lại với LIMIT=None để tải phần còn lại và tiếp tục checkpoint.
Không xóa ảnh đã tải giữa các lần resume vì script kiểm tra size/mtime.
Không tự upload kết quả hoặc xóa dữ liệu. Giữ nguyên cấu hình khi resume.


In [ ]:
for repo_id in REPOS:
    repo_name = repo_id.rsplit('/', 1)[1]
    output = OUTPUT_ROOT / repo_name
    output.mkdir(parents=True, exist_ok=True)
    provenance = output / 'hf-source.json'
    if provenance.exists():
        source_info = json.loads(provenance.read_text())
        assert source_info['repo_id'] == repo_id
        revision = source_info['revision']
    else:
        revision = api.repo_info(repo_id, repo_type='dataset').sha
        provenance.write_text(json.dumps({'repo_id': repo_id, 'revision': revision}, indent=2))
    files = api.list_repo_files(repo_id, repo_type='dataset', revision=revision)
    images = sorted(p for p in files if Path(p).suffix.lower() in {'.jpg', '.jpeg', '.png'}
                    and fnmatch.fnmatchcase(p, IMAGE_PATTERN))
    if not images:
        print('Ví dụ file trong repo:', files[:20])
        raise RuntimeError(f'{repo_id}: không có ảnh rời phù hợp. Kiểm tra pattern/Parquet/ZIP.')
    selected = images if LIMIT is None else images[:LIMIT]
    print(repo_id, 'revision=', revision, 'ảnh phù hợp=', len(images), 'sẽ tải=', len(selected))
    local = DOWNLOAD_ROOT / repo_name
    snapshot_download(repo_id, repo_type='dataset', revision=revision,
                      local_dir=str(local), allow_patterns=selected,
                      token=os.environ['HF_TOKEN'], max_workers=8)
    command = [sys.executable, str(SCRIPT), '--source', str(local),
               '--weights', str(WEIGHTS), '--output', str(output),
               '--device', '0', '--batch', str(BATCH), '--imgsz', '1024',
               '--timezone', TIMEZONE,
               '--classes', *map(str, VEHICLE_CLASSES)]
    # Không truyền --limit: thư mục local chỉ có ảnh đã chọn tải; lần chạy sau bỏ qua checkpoint.
    subprocess.run(command, check=True)
    print('Hoàn thành:', output / 'counts.csv')


## Xem bounding box YOLO trên ảnh thực tế

Chạy sau bước benchmark 1.000 ảnh để kiểm tra trước khi inference toàn bộ.
Chọn tối đa 6 ảnh hợp lệ, rải đều trên các camera đã tải. Có thể điền PREVIEW_PATHS
để xem chính xác ảnh mong muốn (ban đêm, đông xe...). Preview dùng cùng tham số
imgsz/conf/iou/max_det và class với script; không chỉnh threshold riêng khi so sánh.
Ảnh vẽ box được hiển thị và lưu dưới yolo-results/previews, có trong ZIP kết quả.


In [ ]:
import hashlib
from collections import Counter
from PIL import Image
from IPython.display import display

PREVIEW_COUNT = 6
PREVIEW_PATHS = []  # Ví dụ [Path('/kaggle/working/traffic-downloads/.../20261112_120000.jpg')]
PREVIEW_DIR = OUTPUT_ROOT / 'previews'
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

# Chọn từ các ảnh inference thành công, không quét/lưu toàn bộ danh sách ảnh vào RAM.
if PREVIEW_PATHS:
    preview_paths = [Path(p) for p in PREVIEW_PATHS][:PREVIEW_COUNT]
else:
    import sqlite3
    from collections import deque
    camera_samples = []
    for repo_id in REPOS:
        repo_name = repo_id.rsplit('/', 1)[1]
        database = OUTPUT_ROOT / repo_name / 'counts.sqlite'
        if not database.exists():
            continue
        with sqlite3.connect(database) as db:
            cameras = db.execute("SELECT DISTINCT camera_id FROM images WHERE status='ok' ORDER BY camera_id").fetchall()
            for (camera,) in cameras:
                rows = db.execute("SELECT path FROM images WHERE status='ok' AND camera_id=? ORDER BY timestamp LIMIT ?",
                                  (camera, PREVIEW_COUNT)).fetchall()
                camera_samples.append(deque(DOWNLOAD_ROOT / repo_name / row[0] for row in rows))
    preview_paths = []
    while camera_samples and len(preview_paths) < PREVIEW_COUNT:
        for queue in camera_samples:
            if queue and len(preview_paths) < PREVIEW_COUNT:
                preview_paths.append(queue.popleft())
        camera_samples = [queue for queue in camera_samples if queue]

assert preview_paths, 'Chạy benchmark trước hoặc điền PREVIEW_PATHS.'
preview_model = YOLO(str(WEIGHTS))
try:
    for image_path in preview_paths:
        result = preview_model.predict(
            source=str(image_path), device=0, imgsz=1024, conf=0.25, iou=0.7,
            max_det=1000, classes=VEHICLE_CLASSES, half=False,
            verbose=False, save=False,
        )[0]
        counts = Counter(int(i) for i in result.boxes.cls.cpu().tolist())
        print(image_path.parent.name, image_path.name,
              'Tổng xe:', sum(counts.values()),
              {result.names[i]: count for i, count in counts.items()})
        # result.plot() trả ảnh BGR; chuyển RGB trước khi hiển thị/lưu với Pillow.
        annotated = Image.fromarray(result.plot(labels=True, conf=True, line_width=2)[..., ::-1])
        suffix = hashlib.sha256(str(image_path).encode()).hexdigest()[:10]
        output_path = PREVIEW_DIR / f'{image_path.parent.name}_{image_path.stem}_{suffix}.jpg'
        annotated.save(output_path, quality=95)
        display(annotated)
finally:
    del preview_model
    torch.cuda.empty_cache()
print('Đã lưu preview:', PREVIEW_DIR)


## Xem và lưu kết quả

counts.csv chứa số đếm từng ảnh, chưa resample thành tensor STGCN.
counts.sqlite là checkpoint cần giữ để resume. Ảnh lỗi có counts trống, không phải 0 xe.
Nếu hai repo trùng camera/timestamp, không cộng hai số đếm; kiểm tra xung đột trước khi ghép.

Trước khi kết thúc phiên: lưu/download thư mục yolo-results (hoặc ZIP bên dưới).
Filesystem tạm có thể mất khi runtime kết thúc. Để resume phiên mới, khôi phục cả output
và dữ liệu ảnh với metadata ban đầu; nếu không giữ được ảnh/mtime, dùng output mới theo
camera/ngày chưa xử lý thay vì tái dùng checkpoint cũ trên ảnh tải lại.
Với dữ liệu lớn, chọn IMAGE_PATTERN theo camera/ngày và output riêng cho từng phần.


In [ ]:
import csv
for csv_path in OUTPUT_ROOT.glob('*/counts.csv'):
    print(csv_path)
    with csv_path.open() as f:
        reader = csv.DictReader(f)
        for _, row in zip(range(3), reader):
            print(row)


In [ ]:
# Chạy sau khi inference đã dừng. ZIP chỉ chứa kết quả, không chứa ảnh thô.
import shutil
archive = shutil.make_archive('/kaggle/working/yolo-results', 'zip', OUTPUT_ROOT)
from IPython.display import FileLink, display
display(FileLink(archive))


Tài liệu API: https://huggingface.co/docs/huggingface_hub/en/package_reference/file_download
và https://www.kaggle.com/discussions/product-feedback/114053
